# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets available in the dataset by their @id and name
print("Available Record Sets:")
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"@id: {record_set.id}, name: {record_set.name}, description: {record_set.description}")
    record_set_ids.append(record_set.id)

# List fields (i.e., columns) for each record set
for record_set in dataset.metadata.record_sets:
    print(f"\nFields for Record Set '@id': {record_set.id}")
    for field in record_set.fields:
        col_ids = ', '.join([col.id for col in getattr(field, 'columns', [])]) if hasattr(field, 'columns') else None
        print(f"  Field: @id={field.id}, name={field.name}, dataType={field.data_type}, columns=[{col_ids}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: Load all records from each record set into pandas DataFrames
dataframes = {}

# For demonstration, extract all record sets (usually, for large datasets, select one or two)
for record_set_id in record_set_ids:
    print(f"Loading records from Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for Record Set {record_set_id}: {df.columns.tolist()}")
    print(df.head(2))  # Show first 2 records

# For remainder of this notebook, we'll operate on the first record set.
target_record_set_id = record_set_ids[0] if record_set_ids else None
target_df = dataframes[target_record_set_id] if target_record_set_id else pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Note:** All fields are referenced by their `@id` value.

In [ ]:
import numpy as np

if target_df.shape[0] > 0:
    # Try to automatically pick a numeric field for demonstration
    first_record_set = None
    for rs in dataset.metadata.record_sets:
        if rs.id == target_record_set_id:
            first_record_set = rs
            break
    numeric_field_id = None
    for field in getattr(first_record_set, 'fields', []):
        if getattr(field, 'data_type', '').lower() in ['float', 'integer', 'number']:
            if field.id in target_df.columns and np.issubdtype(target_df[field.id].dropna().dtype, np.number):
                numeric_field_id = field.id
                break
    if not numeric_field_id:
        # Fallback: Try to auto-detect a numeric column from dataframe
        numeric_candidates = target_df.select_dtypes(include=[np.number]).columns.tolist()
        numeric_field_id = numeric_candidates[0] if numeric_candidates else None

    if numeric_field_id:
        print(f"Using numeric field for analysis: {numeric_field_id}")
        # Drop NaNs for numeric analysis
        clean_df = target_df.dropna(subset=[numeric_field_id]).copy()
        threshold = clean_df[numeric_field_id].mean() # for demonstration purposes, use mean as threshold
        filtered_df = clean_df[clean_df[numeric_field_id] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records from {len(clean_df)} based on {numeric_field_id} > {threshold:.4f}")
        # Add normalized column
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a string/categorical field
        group_field_id = None
        for field in getattr(first_record_set, 'fields', []):
            if getattr(field, 'data_type', '').lower() in ['string', 'text']:
                if field.id in filtered_df.columns:
                    group_field_id = field.id
                    break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for demonstration.")
else:
    print("No records available in the first record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show histogram for the selected numeric field, if available
if target_df.shape[0] > 0 and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(target_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If we found a group field, show boxplot
    if group_field_id:
        plt.figure(figsize=(12, 5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=60)
        plt.show()
else:
    print("Not enough data or no numeric field for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR\u2072 protein dataset using `mlcroissant`.
- We previewed dataset metadata, record sets, and the fields (referenced by their `@id`).
- Data was loaded into pandas DataFrames, and numeric features were processed and visualized.
- All operations referenced Croissant entity identifiers (`@id`) for maximum reproducibility.

For in-depth analysis, consult the dataset's Croissant schema and documentation for detailed descriptions of record set and field semantics.